# Add Users to a Tableau Server Group

This notebook uses the **Tableau Server Client (TSC)** library to:
1. Authenticate via Personal Access Token
2. Find a target group by name
3. Read a list of **user IDs** from a CSV file
4. Check which users are already in the group
5. Add missing users to the group

**Docs:** [TSC Groups API Reference](https://tableau.github.io/server-client-python/docs/api-ref#groups)

## 0 — Install Dependencies

Run this cell once if you don't already have `tableauserverclient` installed.

In [ ]:
# Uncomment the line below to install
# !pip install tableauserverclient

## 1 — Configuration

Update these variables with your Tableau Server details.

In [ ]:
# ---------- UPDATE THESE ----------

SERVER_URL        = "https://your-tableau-server.com"   # Tableau Server or Cloud URL
TOKEN_NAME        = "your-token-name"                   # Personal Access Token name
TOKEN_VALUE       = "your-token-value"                  # Personal Access Token value
SITE_ID           = ""                                  # Site content URL ("" = default site)
TARGET_GROUP_NAME = "My Target Group"                   # Group to add users to
CSV_FILE_PATH     = "users_to_add.csv"                  # CSV with a "user_id" column

## 2 — Imports & Helper Functions

In [ ]:
import csv
import sys
import tableauserverclient as TSC


def get_all_items(endpoint_get_method):
    """
    Handles TSC pagination so you get ALL items, not just the first page.

    TSC's .get() returns a max of 100 items per call by default.
    This loops through every page and collects everything.

    Parameters
    ----------
    endpoint_get_method : callable
        A TSC endpoint's .get method, e.g. server.groups.get

    Returns
    -------
    list
        All items from every page of results.
    """
    all_items = []
    req_options = TSC.RequestOptions()
    req_options.pagesize = 1000

    all_items, pagination = endpoint_get_method(req_options)
    all_items = list(all_items)

    while len(all_items) < pagination.total_available:
        req_options.pagenumber += 1
        next_page, pagination = endpoint_get_method(req_options)
        all_items.extend(list(next_page))

    return all_items


def read_user_ids_from_csv(filepath):
    """
    Reads Tableau user IDs from a CSV file.

    Expects a CSV with at least a "user_id" column header.
    Strips whitespace and skips blank rows.

    Parameters
    ----------
    filepath : str
        Path to the CSV file.

    Returns
    -------
    list[str]
        List of user ID strings.
    """
    user_ids = []
    with open(filepath, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        if "user_id" not in reader.fieldnames:
            raise ValueError(f"CSV must have a 'user_id' column. Found: {reader.fieldnames}")

        for row in reader:
            uid = row["user_id"].strip()
            if uid:
                user_ids.append(uid)

    return user_ids


print("Helpers loaded!")

## 3 — Authenticate

Creates a server connection and signs in using your PAT.  
We keep the `server` object alive so subsequent cells can use it.

In [ ]:
auth = TSC.PersonalAccessTokenAuth(TOKEN_NAME, TOKEN_VALUE, site_id=SITE_ID)
server = TSC.Server(SERVER_URL, use_server_version=True)

# Sign in — we'll sign out manually in the last cell
server.auth.sign_in(auth)
print(f"Signed in to {SERVER_URL} successfully!")

## 4 — Find the Target Group

In [ ]:
print(f"Looking for group: '{TARGET_GROUP_NAME}' ...")

all_groups = get_all_items(server.groups.get)

target_group = None
for group in all_groups:
    if group.name == TARGET_GROUP_NAME:
        target_group = group
        break

if target_group is None:
    raise ValueError(f"Group '{TARGET_GROUP_NAME}' not found. Available: {[g.name for g in all_groups]}")

print(f"Found group: '{target_group.name}' (ID: {target_group.id})")

## 5 — Load User IDs from CSV

Your CSV should look like this:

```
user_id
a1b2c3d4-e5f6-7890-abcd-ef1234567890
f9e8d7c6-b5a4-3210-fedc-ba9876543210
```

In [ ]:
user_ids_to_add = read_user_ids_from_csv(CSV_FILE_PATH)

print(f"Found {len(user_ids_to_add)} user IDs in '{CSV_FILE_PATH}':")
for uid in user_ids_to_add:
    print(f"  - {uid}")

## 6 — Check Existing Group Members

Pre-loads current members so we can skip users who are already in the group.  
This makes the script **idempotent** — safe to re-run without errors.

In [ ]:
print("Checking existing group members ...")

server.groups.populate_users(target_group)
existing_member_ids = {user.id for user in target_group.users}

print(f"Group '{target_group.name}' currently has {len(existing_member_ids)} members.")

## 7 — Add Users to Group

Loops through the user IDs from the CSV and adds each one.  
Each user gets tagged as `[ADDED]`, `[SKIPPED]`, or `[ERROR]`.

In [ ]:
added = 0
skipped = 0
errors = 0

print("=" * 50)
print("Processing users ...")
print("=" * 50)

for user_id in user_ids_to_add:
    if user_id in existing_member_ids:
        print(f"  [SKIPPED]    '{user_id}' — already in group")
        skipped += 1
        continue

    try:
        server.groups.add_user(target_group, user_id)
        print(f"  [ADDED]      '{user_id}'")
        added += 1
        existing_member_ids.add(user_id)
    except Exception as e:
        print(f"  [ERROR]      '{user_id}' — {e}")
        errors += 1

print()
print("=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"  Added:   {added}")
print(f"  Skipped: {skipped} (already in group)")
print(f"  Errors:  {errors}")
print(f"  Total:   {len(user_ids_to_add)}")

## 8 — Sign Out

Always sign out when you're done to invalidate the auth token.

In [ ]:
server.auth.sign_out()
print("Signed out. Done!")